# Fan-out join · **02 — Reconciliation gate**

**Nothing downstream runs until this passes.** The last cell raises if any
check fails, so `jupyter nbconvert --execute` exits non-zero.

The fan-out in notebook 03 is only worth writing about if the figure it
distorts is the right one. So before I break anything, I prove that the
correct-grain query over my star schema reproduces NHS England's published
August 2026 figures **by treatment function**: the breakdown the chart uses.
Every target is read out of NHS England's own files, never typed in.

| gate | target | what must hold |
|---|---|---|
| **T** | press notice Table 1 (England by treatment function, not including estimates) | pathways exact for all 23 functions and the total; % within 18 weeks to Table 1's one decimal place |
| **P** | Incomplete Provider workbook (NHS and independent-sector sheets) | every provider × treatment function: total, within 18, over 52 and **every one of the 105 week bands**, exact |

This is the same measure as the verification pipeline's Gate A (reported
providers, NONC excluded). That gate checked provider totals; this one goes
a level down, to the treatment function.

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd
import pypdf

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import BRONZE_DIR, GOLD_DIR, STAR_DIR, TOTAL_TF_CODE  # noqa: E402

OUT = GOLD_DIR / "00_reconciliation"
OUT.mkdir(parents=True, exist_ok=True)
PUB = BRONZE_DIR / "nhse_rtt_published_2026-08"
pd.set_option("display.width", 200)

fact = pd.read_parquet(STAR_DIR / "fact_incomplete_pathways.parquet")
tf = pd.read_parquet(STAR_DIR / "dim_treatment_function.parquet")
wb_ = pd.read_parquet(STAR_DIR / "dim_week_band.parquet")

checks = []
def check(gate, name, published, rebuilt, ok, note=""):
    checks.append({"gate": gate, "check": name, "published": published, "rebuilt": rebuilt,
                   "pass": bool(ok), "note": note})

## The correct-grain query

This is the query notebook 03 reuses as its baseline: fact → treatment
function and week band dimensions, both many-to-one, then sum. Nothing
else.

In [2]:
def by_treatment_function(f: pd.DataFrame) -> pd.DataFrame:
    x = (f.merge(wb_[["week_band", "within_18"]], on="week_band", validate="many_to_one")
          .merge(tf, on="tf_code", validate="many_to_one"))
    x["within_18_pathways"] = x.pathways.where(x.within_18, 0)
    g = x.groupby(["tf_code", "tf_name"], as_index=False)[["pathways", "within_18_pathways"]].sum()
    g["pct_within_18"] = 100 * g.within_18_pathways / g.pathways
    g["pct_over_18"] = 100 - g.pct_within_18
    return g

correct = by_treatment_function(fact)
correct.sort_values("pathways", ascending=False)

,tf_code,tf_name,pathways,within_18_pathways,pct_within_18,pct_over_18
2,C_110,Trauma and Orthopaedic Service,843419,515333,61.100473,38.899527
4,C_130,Ophthalmology Service,622784,449910,72.241740,27.758260
18,X02,Other - Medical Services,620378,434923,70.106129,29.893871
3,C_120,Ear Nose and Throat Service,589316,334550,56.769204,43.230796
17,C_502,Gynaecology Service,585378,348917,59.605417,40.394583
21,X05,Other - Surgical Services,473107,316505,66.899243,33.100757
12,C_330,Dermatology Service,422625,291772,69.038036,30.961964
0,C_100,General Surgery Service,384427,242873,63.177925,36.822075
1,C_101,Urology Service,366130,237447,64.853194,35.146806
11,C_320,Cardiology Service,360799,244937,67.887383,32.112617


## Gate T — press notice Table 1

I pull Table 1 out of the PDF text, anchored on its own title, and parse
each row as *name, pathways, % within 18, then three completed / new
period counts* that I don't need. If the table doesn't parse into exactly
23 functions plus a total, the notebook stops.

In [3]:
text = re.sub(r"\s+", " ", " ".join(p.extract_text() for p in pypdf.PdfReader(
    PUB / "Aug26-RTT-statistical-press-notice-PDF-389K-vjgRdG.pdf").pages))
start = text.index("Table 1 – RTT pathways by treatment function, not including estimates for missing trusts, August 2026, England")
end = text.index("Table 2 – RTT waiting times time series", start)
table1 = text[start:end]
rows = re.findall(r"([A-Z][A-Za-z \-]+?) ([\d,]+) ([\d.]+)% [\d,]+ [\d,]+ [\d,]+", table1)
t1 = pd.DataFrame(rows, columns=["tf_name", "pathways", "pct_within_18"])
t1["tf_name"] = t1.tf_name.str.replace(r"^.*?Total\s+(?=\S)", "", regex=True).str.strip()
t1["pathways"] = t1.pathways.str.replace(",", "").astype("int64")
t1["pct_within_18"] = t1.pct_within_18.astype(float)
assert len(t1) == 24 and t1.tf_name.iloc[-1] == "Total", t1.tf_name.tolist()
t1

,tf_name,pathways,pct_within_18
0,General Surgery Service,384427,63.2
1,Urology Service,366130,64.9
2,Trauma and Orthopaedic Service,843419,61.1
3,Ear Nose and Throat Service,589316,56.8
4,Ophthalmology Service,622784,72.2
5,Oral Surgery Service,315652,53.8
6,Neurosurgical Service,53980,62.0
7,Plastic Surgery Service,108674,59.5
8,Cardiothoracic Surgery Service,8551,73.9
9,General Internal Medicine Service,28907,74.2


In [4]:
m = t1[t1.tf_name != "Total"].merge(correct, on="tf_name", how="outer", suffixes=("_pub", ""), indicator=True)
assert (m._merge == "both").all(), m[m._merge != "both"]
m["pct_rounded"] = m.pct_within_18.round(1)
for r in m.itertuples():
    check("T", f"{r.tf_name}: pathways", int(r.pathways_pub), int(r.pathways), r.pathways_pub == r.pathways)
    check("T", f"{r.tf_name}: % within 18 (1 d.p.)", r.pct_within_18_pub, r.pct_rounded, r.pct_within_18_pub == r.pct_rounded)
tot_pub = t1.set_index("tf_name").loc["Total"]
eng_pct = round(100 * correct.within_18_pathways.sum() / correct.pathways.sum(), 1)
check("T", "England: pathways", int(tot_pub.pathways), int(correct.pathways.sum()), tot_pub.pathways == correct.pathways.sum())
check("T", "England: % within 18 (1 d.p.)", tot_pub.pct_within_18, eng_pct, tot_pub.pct_within_18 == eng_pct)

## Gate P — provider × treatment function, band by band

The workbook lists every treatment function for every provider, zeros
included; my fact table has no zero cells. So I outer-join and treat a
missing row on either side as zero. A national figure can agree by
accident; 533 providers × 23 functions × 105 bands agreeing at once can't.

In [5]:
def band_week(col) -> int | None:
    c = str(col)
    if c == "104 plus":
        return 104
    mm = re.fullmatch(r">(\d+)-(\d+)", c)
    return int(mm.group(1)) if mm else None

wbk = []
for sheet in ["Provider", "IS Provider"]:
    d = pd.read_excel(PUB / "Incomplete-Provider-Aug26-XLSX-9M-vjgRdG.xlsx", sheet_name=sheet, header=13)
    d = d[d["Treatment Function Code"].notna() & (d["Treatment Function Code"] != TOTAL_TF_CODE)]
    wbk.append(d.assign(sheet=sheet))
wbk = pd.concat(wbk, ignore_index=True)
band_cols = {c: band_week(c) for c in wbk.columns if band_week(c) is not None}
assert sorted(band_cols.values()) == list(range(105)), "workbook band columns incomplete"
assert not wbk.duplicated(["Provider Code", "Treatment Function Code"]).any()

pub_long = (wbk.rename(columns={"Provider Code": "provider_code", "Treatment Function Code": "tf_code", **band_cols})
               .melt(id_vars=["provider_code", "tf_code"], value_vars=list(range(105)),
                     var_name="week_band", value_name="pathways"))
pub_long["pathways"] = pd.to_numeric(pub_long.pathways, errors="coerce").fillna(0).astype("int64")
mine_long = fact.groupby(["provider_code", "tf_code", "week_band"], as_index=False).pathways.sum()

cell = pub_long.merge(mine_long, on=["provider_code", "tf_code", "week_band"], how="outer",
                      suffixes=("_pub", "")).fillna(0)
bad_cells = cell[cell.pathways_pub != cell.pathways]
check("P", "every provider x TF x week band", int(cell.pathways_pub.sum()), int(cell.pathways.sum()),
      bad_cells.empty, f"{len(cell):,} cells compared; {len(bad_cells)} differ")

# The workbook's own derived columns, against the same cells summed my way.
pt = cell.merge(wb_[["week_band", "within_18", "over_52"]], on="week_band")
mine_pt = pt.assign(w18=pt.pathways.where(pt.within_18, 0), o52=pt.pathways.where(pt.over_52, 0)) \
            .groupby(["provider_code", "tf_code"])[["pathways", "w18", "o52"]].sum()
pub_pt = wbk.set_index(["Provider Code", "Treatment Function Code"])[
    ["Total number of incomplete pathways", "Total within 18 weeks", "Total 52 plus weeks"]]
pub_pt.index.names = ["provider_code", "tf_code"]
pub_pt.columns = ["pathways", "w18", "o52"]
both = pub_pt.apply(pd.to_numeric, errors="coerce").fillna(0).join(mine_pt, rsuffix="_mine", how="outer").fillna(0)
for c, label in [("pathways", "total"), ("w18", "within 18"), ("o52", "over 52")]:
    bad = both[both[c] != both[f"{c}_mine"]]
    check("P", f"every provider x TF: {label}", int(both[c].sum()), int(both[f"{c}_mine"].sum()),
          bad.empty, f"{len(both):,} provider x TF rows; {len(bad)} differ")
n_prov = fact.provider_code.nunique()
check("P", "providers in my fact = providers with pathways in the workbook",
      int((pub_pt.groupby(level=0).pathways.sum() > 0).sum()), n_prov,
      (pub_pt.groupby(level=0).pathways.sum() > 0).sum() == n_prov)

/tmp/ipykernel_909/609230939.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  wbk.append(d.assign(sheet=sheet))


/tmp/ipykernel_909/609230939.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  wbk.append(d.assign(sheet=sheet))


## The gate

In [6]:
gate = pd.DataFrame(checks)
gate.to_csv(OUT / "reconciliation.csv", index=False)
correct.to_csv(OUT / "correct_grain_by_tf.csv", index=False)
print(gate.groupby("gate")["pass"].agg(["sum", "count"]).rename(columns={"sum": "passed", "count": "checks"}))
gate[~gate["pass"]] if (~gate["pass"]).any() else gate[gate.gate == "P"]

      passed  checks
gate                
P          5       5
T         48      48


,gate,check,published,rebuilt,pass,note
48,P,every provider x TF x week band,7158164.0,7158164.0,True,"1,287,195 cells compared; 0 differ"
49,P,every provider x TF: total,7158164.0,7158164.0,True,"12,259 provider x TF rows; 0 differ"
50,P,every provider x TF: within 18,4667554.0,4667554.0,True,"12,259 provider x TF rows; 0 differ"
51,P,every provider x TF: over 52,109936.0,109936.0,True,"12,259 provider x TF rows; 0 differ"
52,P,providers in my fact = providers with pathways...,532.0,532.0,True,


In [7]:
# I raise rather than print, so a failure stops a scripted run.
failed = gate[~gate["pass"]]
if len(failed):
    print(failed.to_string())
    raise SystemExit(f"GATE: FAIL ({len(failed)} of {len(gate)} checks)")
print(f"GATE: PASS ({len(gate)} of {len(gate)} checks)")

GATE: PASS (53 of 53 checks)
